# Residue Class Graduation Analysis

Investigates whether the anti-diagonal graduation structure predicted by residue-class
learning is present — and at what granularity — in four reference models:

| Model | Description |
|-------|-------------|
| p109/s485/ds598 | Reference healthy — clear anti-diagonal banding, sharp specialization |
| p113/s999/ds598 | Canon model — diffuse specialization, no anti-diagonal banding |
| p59/s485/ds598  | Incomplete frequency set — disorganized graduation map |
| p101/s999/ds598 | High final accuracy (97.6%) but no anti-diagonal structure |

**Hypothesis being tested:** If a model learns by residue class, pairs with the same
`(a+b) mod p` should graduate at similar epochs. The within-class graduation spread
(std of graduation epochs per residue class) captures this directly.

**Key questions:**
1. Do p113 and p101 differ in within-class spread despite both showing no anti-diagonal structure?
2. Is there a systematic ordering of residue classes (some graduating earlier than others)?
3. Does the GCD blind-spot prediction hold — classes at multiples of committed frequencies should graduate together (lower spread)?

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from miscope import load_family

In [2]:
FAMILY_NAME = "modulo_addition_1layer"
EXPORT_DIR = Path("exports/residue_graduation")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# (prime, model_seed, data_seed, label)
MODELS = [
    (109, 485, 598, "p109 reference healthy"),
    (113, 999, 598, "p113 canon (diffuse)"),
    (59,  485, 598, "p59 incomplete freq set"),
    (101, 999, 598, "p101 high-acc no structure"),
]

family = load_family(FAMILY_NAME)

## Load and summarize graduation artifacts

In [3]:
def load_graduation_data(family, prime, seed, data_seed):
    """Load cross_epoch graduation artifact for one variant."""
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    data = v.artifacts.load_cross_epoch("input_trace_graduation")
    return data


def compute_class_stats(data, p):
    """Per-residue-class graduation stats for test pairs.

    Returns a dict of arrays indexed by residue class c (0..p-1):
        mean_epoch   float (p,)  — mean graduation epoch; nan if no test pairs
        std_epoch    float (p,)  — std graduation epoch; nan if fewer than 2 test pairs
        median_epoch float (p,)  — median graduation epoch
        n_pairs      int   (p,)  — count of test pairs in class c
        n_graduated  int   (p,)  — count that graduated (epoch >= 0)
        pair_epochs  list        — list of p arrays of graduation epochs (test, graduated only)
    """
    ge = data["graduation_epochs"]  # (p²,) int32; -1 = never graduated
    split = data["split"]           # (p²,) bool; True = train

    # Probe order: k -> a = k // p, b = k % p
    a_all = np.arange(p * p) // p
    b_all = np.arange(p * p) % p
    residue = (a_all + b_all) % p

    test_mask = ~split

    mean_epoch   = np.full(p, np.nan)
    std_epoch    = np.full(p, np.nan)
    median_epoch = np.full(p, np.nan)
    n_pairs      = np.zeros(p, dtype=int)
    n_graduated  = np.zeros(p, dtype=int)
    pair_epochs  = []

    for c in range(p):
        class_test = test_mask & (residue == c)
        n_pairs[c] = class_test.sum()

        graduated_mask = class_test & (ge >= 0)
        n_graduated[c] = graduated_mask.sum()
        epochs_c = ge[graduated_mask].astype(float)
        pair_epochs.append(epochs_c)

        if len(epochs_c) > 0:
            mean_epoch[c]   = epochs_c.mean()
            median_epoch[c] = np.median(epochs_c)
        if len(epochs_c) > 1:
            std_epoch[c] = epochs_c.std()

    return {
        "mean_epoch":   mean_epoch,
        "std_epoch":    std_epoch,
        "median_epoch": median_epoch,
        "n_pairs":      n_pairs,
        "n_graduated":  n_graduated,
        "pair_epochs":  pair_epochs,
    }


models_data = {}
for prime, seed, data_seed, label in MODELS:
    data = load_graduation_data(family, prime, seed, data_seed)
    stats = compute_class_stats(data, prime)
    models_data[(prime, seed, data_seed)] = {
        "data": data, "stats": stats, "label": label, "prime": prime
    }
    ge = data["graduation_epochs"]
    split = data["split"]
    test_ge = ge[~split]
    print(f"{label}:")
    print(f"  test pairs: {test_ge.shape[0]}, never-graduated: {(test_ge == -1).sum()}")
    print(f"  graduation epoch range: [{test_ge[test_ge >= 0].min()}, {test_ge[test_ge >= 0].max()}]")
    print(f"  mean within-class spread (std): {np.nanmean(stats['std_epoch']):.0f} epochs")
    print()

p109 reference healthy:
  test pairs: 8317, never-graduated: 0
  graduation epoch range: [0, 4800]
  mean within-class spread (std): 1063 epochs

p113 canon (diffuse):
  test pairs: 8939, never-graduated: 0
  graduation epoch range: [100, 11300]
  mean within-class spread (std): 1608 epochs

p59 incomplete freq set:
  test pairs: 2437, never-graduated: 0
  graduation epoch range: [0, 25500]
  mean within-class spread (std): 5278 epochs

p101 high-acc no structure:
  test pairs: 7141, never-graduated: 0
  graduation epoch range: [100, 17000]
  mean within-class spread (std): 3232 epochs



## Within-class graduation spread

Box plot per residue class, sorted by mean graduation epoch.
Tight boxes = residue class graduates as a unit (predicted by frequency-based learning).
Wide boxes = pairs with same residue sum graduate at scattered epochs.

In [ ]:
def render_within_class_spread(prime, seed, data_seed):
    """Box plot of graduation epoch distribution per residue class, sorted by median."""
    entry = models_data[(prime, seed, data_seed)]
    stats = entry["stats"]
    label = entry["label"]
    p = prime

    # Sort classes by median graduation epoch
    valid = ~np.isnan(stats["median_epoch"])
    sort_order = np.argsort(stats["median_epoch"] * valid + (~valid) * 1e9)

    fig = go.Figure()
    for rank, c in enumerate(sort_order):
        epochs_c = stats["pair_epochs"][c]
        if len(epochs_c) == 0:
            continue
        color_h = int(360 * c / p)
        fig.add_trace(
            go.Box(
                y=epochs_c.tolist(),
                name=f"c={c}",
                marker_color=f"hsl({color_h}, 70%, 50%)",
                boxpoints="outliers",
                line_width=1.5,
                showlegend=False,
            )
        )

    fig.update_layout(
        title=f"{label} — within-class graduation spread (sorted by median epoch)",
        xaxis_title="Residue class (sorted by median graduation epoch)",
        yaxis_title="Graduation epoch",
        template="plotly_white",
        height=400,
        margin=dict(l=60, r=20, t=50, b=60),
    )
    return fig


for prime, seed, data_seed, _ in MODELS:
    fig = render_within_class_spread(prime, seed, data_seed)
    fig.show()
    fig.write_image(f"exports/p{prime}s{seed}ds{data_seed}_within_class_graduation_spread.png", format="png")

FileNotFoundError: [Errno 2] No such file or directory: 'exports/p109s485ds598/within_class_graduation_spread.png'

## Cross-model comparison: mean graduation epoch per class

Scatter plot of mean graduation epoch per residue class, one panel per model.
Color encodes residue class (0..p-1) using the same HSL wheel as the accuracy timeline.

If the model learns classes in a structured order (e.g., GCD-related clusters),
the scatter should show groups of classes with similar mean epochs.

In [ ]:
def render_class_ordering_comparison():
    """2×2 scatter grid of mean graduation epoch per residue class, colored by class."""
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[entry["label"] for entry in models_data.values()],
        vertical_spacing=0.14,
        horizontal_spacing=0.10,
    )

    for idx, ((prime, seed, data_seed), entry) in enumerate(models_data.items()):
        row, col = divmod(idx, 2)
        stats = entry["stats"]
        p = prime

        mean_e = stats["mean_epoch"]
        std_e  = stats["std_epoch"]
        colors = [f"hsl({int(360 * c / p)}, 70%, 50%)" for c in range(p)]

        # Sort by mean epoch for x-axis
        valid_classes = np.where(~np.isnan(mean_e))[0]
        sort_idx = valid_classes[np.argsort(mean_e[valid_classes])]

        x_pos = np.arange(len(sort_idx))
        y_vals = mean_e[sort_idx]
        y_err  = std_e[sort_idx]
        marker_colors = [colors[c] for c in sort_idx]
        hover_texts = [f"c={c}<br>mean={mean_e[c]:.0f}<br>std={std_e[c]:.0f}" for c in sort_idx]

        fig.add_trace(
            go.Scatter(
                x=x_pos.tolist(),
                y=y_vals.tolist(),
                mode="markers",
                marker=dict(color=marker_colors, size=8, line=dict(width=0.5, color="white")),
                error_y=dict(type="data", array=y_err.tolist(), visible=True, thickness=1, width=3),
                text=hover_texts,
                hoverinfo="text",
                showlegend=False,
            ),
            row=row + 1, col=col + 1,
        )

    fig.update_xaxes(title_text="Class rank (by mean grad epoch)", row=2)
    fig.update_yaxes(title_text="Mean graduation epoch", col=1)
    fig.update_layout(
        title="Mean graduation epoch per residue class (± 1 std, sorted)",
        template="plotly_white",
        height=600,
        margin=dict(l=60, r=20, t=60, b=60),
    )
    return fig


fig = render_class_ordering_comparison()
fig.show()

## Cross-model spread summary

For each model: distribution of within-class spread values (one value per residue class).
A model with low median spread graduates entire residue classes together.
A model with high spread graduates individual pairs independently of class.

In [ ]:
def render_spread_summary():
    """Violin plot comparing within-class spread distributions across all four models."""
    fig = go.Figure()

    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

    for (prime, seed, data_seed), color in zip(models_data.keys(), colors):
        entry = models_data[(prime, seed, data_seed)]
        std_vals = entry["stats"]["std_epoch"]
        valid_std = std_vals[~np.isnan(std_vals)]

        fig.add_trace(
            go.Violin(
                y=valid_std.tolist(),
                name=entry["label"],
                box_visible=True,
                meanline_visible=True,
                fillcolor=color,
                opacity=0.7,
                line_color=color,
            )
        )

    fig.update_layout(
        title="Within-class graduation spread — distribution across residue classes",
        yaxis_title="Std of graduation epoch (per class)",
        template="plotly_white",
        height=420,
        margin=dict(l=60, r=20, t=50, b=80),
        legend=dict(orientation="h", y=1.12),
        violinmode="overlay",
    )
    return fig


fig = render_spread_summary()
fig.show()

## Staircase detection — inter-class gap vs intra-class spread

A clean staircase requires two properties:
1. **Low intra-class spread** — pairs within a class graduate together
2. **High inter-class gap** — different classes graduate at meaningfully different epochs

This cell computes the ratio: `inter-class gap / intra-class spread` as a signal-to-noise
measure for staircase structure. Higher ratio = more staircase-like.

In [ ]:
def staircase_stats(entry):
    """Return staircase structure metrics for one model."""
    stats = entry["stats"]
    ge = entry["data"]["graduation_epochs"]
    split = entry["data"]["split"]
    test_ge = ge[~split]
    graduated = test_ge[test_ge >= 0]

    mean_e = stats["mean_epoch"]
    std_e  = stats["std_epoch"]

    valid = ~np.isnan(mean_e)
    inter_std   = float(np.std(mean_e[valid])) if valid.sum() > 1 else float("nan")
    intra_median = float(np.nanmedian(std_e)) if not np.all(np.isnan(std_e)) else float("nan")
    snr = inter_std / intra_median if intra_median > 0 else float("nan")

    # Normalized spread: std as fraction of total graduation epoch range for this model
    epoch_range = float(graduated.max() - graduated.min()) if len(graduated) > 1 else 1.0
    intra_norm  = intra_median / epoch_range if epoch_range > 0 else float("nan")
    inter_norm  = inter_std / epoch_range if epoch_range > 0 else float("nan")

    return {
        "inter_std": inter_std,
        "intra_median": intra_median,
        "snr": snr,
        "epoch_range": epoch_range,
        "intra_norm": intra_norm,  # intra-class spread as fraction of total range
        "inter_norm": inter_norm,  # inter-class spread as fraction of total range
    }


print(f"{'Model':<35} {'range':>7} {'inter_std':>10} {'intra_med':>10} {'SNR':>6}  {'intra/range':>12} {'inter/range':>12}")
print("-" * 100)

for (prime, seed, data_seed), entry in models_data.items():
    m = staircase_stats(entry)
    print(
        f"{entry['label']:<35}"
        f" {m['epoch_range']:>7.0f}"
        f" {m['inter_std']:>10.0f}"
        f" {m['intra_median']:>10.0f}"
        f" {m['snr']:>6.2f}"
        f"  {m['intra_norm']:>12.3f}"
        f" {m['inter_norm']:>12.3f}"
    )

print()
print("intra/range: within-class spread as fraction of total graduation window")
print("inter/range: inter-class spread as fraction of total graduation window")
print("Lower intra/range = more cohesive class-level graduation")

## Per-class graduation epoch heatmap

Each row is one residue class, each column is one test pair in that class (sorted by
graduation epoch). Color encodes graduation epoch. Uniform row colors = cohesive class
graduation. Rainbow rows = pairs graduate independently.

In [ ]:
def render_class_cohesion_heatmap(prime, seed, data_seed):
    """Heatmap: rows = residue classes (sorted by mean epoch), cols = pairs within class."""
    entry = models_data[(prime, seed, data_seed)]
    stats = entry["stats"]
    label = entry["label"]
    p = prime

    # Sort classes by mean graduation epoch
    mean_e = stats["mean_epoch"]
    class_order = np.argsort(np.where(np.isnan(mean_e), np.inf, mean_e))

    # Find max class size for consistent column count
    max_class_size = max(len(stats["pair_epochs"][c]) for c in range(p))

    # Build matrix: rows = classes in sorted order, cols = pair rank within class
    matrix = np.full((p, max_class_size), np.nan)
    y_labels = []
    for row_idx, c in enumerate(class_order):
        epochs_c = np.sort(stats["pair_epochs"][c])
        matrix[row_idx, : len(epochs_c)] = epochs_c
        y_labels.append(f"c={c}")

    fig = go.Figure(
        go.Heatmap(
            z=matrix.tolist(),
            colorscale="Viridis",
            colorbar=dict(title="Graduation<br>Epoch"),
            hovertemplate="class=%{y}, pair_rank=%{x}<br>epoch=%{z}<extra></extra>",
        )
    )
    fig.update_layout(
        title=f"{label} — per-pair graduation epochs within residue classes",
        xaxis_title="Pair rank within class (sorted by epoch)",
        yaxis=dict(
            title="Residue class (sorted by mean epoch)",
            tickvals=list(range(p)),
            ticktext=y_labels,
            tickfont=dict(size=9),
        ),
        template="plotly_white",
        height=max(350, p * 14),
        margin=dict(l=80, r=60, t=50, b=60),
    )
    return fig


for prime, seed, data_seed, _ in MODELS:
    fig = render_class_cohesion_heatmap(prime, seed, data_seed)
    fig.show()

## Export figures

In [ ]:
for prime, seed, data_seed, _ in MODELS:
    tag = f"p{prime}_s{seed}_ds{data_seed}"

    fig = render_within_class_spread(prime, seed, data_seed)
    fig.write_image(str(EXPORT_DIR / f"{tag}_within_class_spread.png"))

    fig = render_class_cohesion_heatmap(prime, seed, data_seed)
    fig.write_image(str(EXPORT_DIR / f"{tag}_class_cohesion_heatmap.png"))

    print(f"  exported {tag}")

render_class_ordering_comparison().write_image(str(EXPORT_DIR / "all_class_ordering.png"))
render_spread_summary().write_image(str(EXPORT_DIR / "all_spread_summary.png"))
print("  exported cross-model figures")